# nb_02_silver_transform — cleaned, typed, deduped, DQ applied

Bronze → Silver for one run: flatten and type the raw payload, re-assert the CA/OR/WA scope,
**DQ-Q1 quarantine first** (invalid coordinates → Delta quarantine table, never Silver/Gold),
**then** dedupe by `chapter_id` (highest `source_object_id` wins — a bad duplicate can never
shadow a good record), **then** DQ-W1 warning flags. Writes Silver as a Delta overwrite and
**exits** with the run's count ledger as JSON for the Gold notebook.

`run_id` is the only required lineage parameter — the notebook locates the bronze folder and
reads its metadata itself, so it works identically inside the pipeline or run by hand.

In [ ]:
storage_account = ""          # required — ADLS Gen2 account name
lake_container = "lake"
run_id = ""                   # required — bronze run to transform (from nb_01 exit value)

In [ ]:
%run nb_00_config

In [ ]:
cfg = init_config(storage_account, lake_container)
if not run_id:
    raise PipelineError("Parameter 'run_id' is required (bronze run to transform).")

bronze_dir = find_bronze_run_dir(cfg, run_id)
metadata = read_ingest_metadata(bronze_dir)

raw = spark.read.option("multiLine", "true").json(f"{bronze_dir}/page_*.json")
if "features" not in raw.columns:
    raise PipelineError(f"Bronze payload at {bronze_dir} has no 'features' array")

flat = (
    raw.select(F.explode("features").alias("feature"))
    .select(
        F.trim(F.col("feature.attributes.ChapterID").cast("string")).alias("chapter_id"),
        F.trim(F.col("feature.attributes.University_Chapter").cast("string")).alias("chapter_name"),
        F.trim(F.col("feature.attributes.City").cast("string")).alias("city"),
        F.upper(F.trim(F.col("feature.attributes.State").cast("string"))).alias("state"),
        # try_cast: malformed coordinates must become NULL (-> DQ-Q1), not crash the job
        F.expr("try_cast(feature.geometry.x as double)").alias("longitude"),
        F.expr("try_cast(feature.geometry.y as double)").alias("latitude"),
        F.col("feature.attributes.OBJECTID").cast("long").alias("source_object_id"),
        F.to_json(F.col("feature")).alias("raw_payload"),
    )
    .withColumn("ingest_run_id", F.lit(metadata["run_id"]))
    .withColumn("ingested_at_utc", F.lit(metadata["ingested_at_utc"]).cast("timestamp"))
)

rows_in = flat.count()
scoped = flat.filter(F.col("state").isin(*IN_SCOPE_STATES))
rows_out_of_scope = rows_in - scoped.count()

flagged = with_dq_flags(scoped)

quarantine_df = flagged.filter(F.col("is_quarantined")).select(
    "chapter_id", "chapter_name", "city", "state", "longitude", "latitude",
    "source_object_id", "quarantine_reason", "ingest_run_id", "ingested_at_utc",
    "raw_payload",
)
survivors = flagged.filter(~F.col("is_quarantined"))

# Dedupe AFTER quarantine (a bad duplicate can never shadow a good record).
w = Window.partitionBy("chapter_id").orderBy(
    F.col("source_object_id").desc_nulls_last(), F.col("chapter_name").asc()
)
deduped = survivors.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")

silver_df = deduped.select(
    "chapter_id", "chapter_name", "city", "state", "longitude", "latitude",
    "dq_status", "dq_warnings", "source_object_id", "ingest_run_id", "ingested_at_utc",
)

counts = {
    "run_id": run_id,
    "rows_in": rows_in,
    "rows_out_of_scope": rows_out_of_scope,
    "rows_quarantined": quarantine_df.count(),
    "rows_deduped": survivors.count() - deduped.count(),
    "rows_warned": silver_df.filter(F.col("dq_status") == DQ_STATUS_WARNING).count(),
    "rows_ok": silver_df.filter(F.col("dq_status") == DQ_STATUS_OK).count(),
}

# Quarantine: Delta, append-by-run, partitioned by run for easy inspection.
(quarantine_df.write.format("delta").mode("append")
 .partitionBy("ingest_run_id").save(cfg.quarantine_path))
# Silver: Delta, full overwrite each run (latest snapshot; history via Delta versions).
(silver_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").save(cfg.silver_path))

log.info("Silver written: %s", counts)
exit_value = json.dumps(counts)
print(exit_value)
mssparkutils.notebook.exit(exit_value)